# Spike 06 — Vision-Based RAG with PageIndex + Llama 4 Scout

**Goal:** Use PageIndex's PDF/vision mode to index the full report visually, then use the page references it returns to feed raw page images into Llama 4 Scout (via Groq) for answer generation.

**Time box:** 2 hours

**Question to answer:** Does the vision-based pipeline (PDF → PageIndex reasoning graph → page numbers → vision LLM) produce better answers than our Spike 05 text pipeline?

**Done when:** A query returns specific page numbers from PageIndex, those pages are loaded as images, and Llama 4 Scout produces a grounded answer citing the visual content.

---

### Why this is different from Spike 05

| | Spike 05 (text RAG) | Spike 06 (vision RAG) |
|---|---|---|
| Input to index | OCR'd markdown text | Original PDF pages (images) |
| PageIndex mode | `/markdown` endpoint | `/doc` endpoint (PDF upload) |
| Retrieval output | Text passages | **Page numbers** |
| Answer generation | Groq reads text | **Llama 4 Scout reads page images** |
| Arabic quality | Depends on OCR accuracy | Direct visual reading |
| Tables/charts | Often broken by OCR | Preserved as rendered |

PageIndex builds a **visual reasoning graph** over the PDF — it understands page layout, headings, tables, and cross-references without needing OCR first. When you query it, it returns the page numbers most likely to contain the answer. A vision LLM then reads exactly those pages.

### Architecture
```
PDF file
  └─▶ PageIndex submit_document()       ← vision reasoning graph built here
         └─▶ doc_id
               └─▶ submit_query(doc_id, question)
                     └─▶ retrieval_id
                           └─▶ get_retrieval(retrieval_id)
                                 └─▶ [page 7, page 23, ...]   ← physical_index fields
                                       └─▶ load + compress PNG → JPEG
                                             └─▶ base64 encode (up to 5 pages)
                                                   └─▶ Llama 4 Scout via Groq (vision)
                                                         └─▶ grounded answer
```

### Free-tier constraints
- **PageIndex**: subset PDF only (14 sampled pages, ~9 MB EN / ~3 MB AR)
- **Groq / Llama 4 Scout**: max 5 images per request, 4 MB per image (base64)
- **Image size**: PNGs compressed to JPEG (1024px wide, quality 82) → ~80-120 KB each → well under the 4 MB limit

## Step 1 — Setup

In [1]:
import re
import json
import base64
import fitz        # PyMuPDF
import time
import os
import io
from pathlib import Path
from dotenv import load_dotenv
from pageindex import PageIndexClient
from groq import Groq
from PIL import Image   # pip install Pillow — already a transitive dependency

load_dotenv(dotenv_path="../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")

if not PAGEINDEX_API_KEY:
    raise ValueError("PAGEINDEX_API_KEY not found in .env")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

pi_client   = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

# Llama 4 Scout: free-tier Groq model with vision support
# Limit: 1 image per request on the free tier — we send the most relevant page only
VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

# Paths
DATA_DIR   = Path("../data")
PDF_DIR    = DATA_DIR / "pdfs"
IMG_EN_DIR = DATA_DIR / "images_en"
IMG_AR_DIR = DATA_DIR / "images_ar"
DOC_CACHE  = DATA_DIR / "vision_doc_ids.json"
EN_PDF     = PDF_DIR / "tranquil_en.pdf"
AR_PDF     = PDF_DIR / "tranquil_ar.pdf"
EN_SUBSET  = DATA_DIR / "tranquil_en_subset.pdf"
AR_SUBSET  = DATA_DIR / "tranquil_ar_subset.pdf"

SAMPLED_PAGES   = [7, 8, 10, 11, 20, 37, 38, 53, 60, 69, 70, 83, 107, 109]
SUBSET_PAGE_MAP = {i + 1: orig for i, orig in enumerate(SAMPLED_PAGES)}

for p in [EN_PDF, AR_PDF, IMG_EN_DIR, IMG_AR_DIR]:
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p} — run spike_01 first")

print("✅ Setup ready")
print(f"   Vision model  : {VISION_MODEL}")
print(f"   Sampled pages : {SAMPLED_PAGES}")

✅ Setup ready
   Vision model  : meta-llama/llama-4-scout-17b-16e-instruct
   Sampled pages : [7, 8, 10, 11, 20, 37, 38, 53, 60, 69, 70, 83, 107, 109]


## Step 2 — Extract Sampled Pages into Small PDFs

We don't upload the full 112-page PDFs (100 MB + 34 MB) — that would blow through the free tier.  
Instead we carve out only the **14 sampled pages** we OCR'd in Spikes 02 & 03 into a small PDF per language.

These are the same pages whose content is already in our PageIndex reasoning graph,  
so the vision answers are directly comparable to the Spike 05 text baseline.

> PyMuPDF (`fitz`) is already installed from Spike 01 — no new dependencies.

In [2]:
def extract_pages_to_pdf(src_pdf: Path, page_numbers: list, out_pdf: Path) -> Path:
    """Extract specific 1-based page numbers from src_pdf into a new small PDF."""
    src  = fitz.open(str(src_pdf))
    dest = fitz.open()
    for page_num in page_numbers:
        idx = page_num - 1
        if 0 <= idx < len(src):
            dest.insert_pdf(src, from_page=idx, to_page=idx)
        else:
            print(f"  ⚠️  Page {page_num} out of range — skipped")
    dest.save(str(out_pdf))
    src.close()
    dest.close()
    size_kb = out_pdf.stat().st_size // 1024
    print(f"  ✅ {out_pdf.name} — {len(page_numbers)} pages, {size_kb} KB")
    return out_pdf


# Only extract if the subset PDFs don't already exist
if EN_SUBSET.exists() and AR_SUBSET.exists():
    print(f"⏭  Subset PDFs already exist — skipping extraction")
    print(f"   EN: {EN_SUBSET.stat().st_size // 1024} KB")
    print(f"   AR: {AR_SUBSET.stat().st_size // 1024} KB")
else:
    print(f"Extracting pages {SAMPLED_PAGES} from each PDF...\n")
    extract_pages_to_pdf(EN_PDF, SAMPLED_PAGES, EN_SUBSET)
    extract_pages_to_pdf(AR_PDF, SAMPLED_PAGES, AR_SUBSET)
    print(f"\nFull EN PDF  : {EN_PDF.stat().st_size  // 1024 // 1024} MB  →  Subset: {EN_SUBSET.stat().st_size // 1024} KB")
    print(f"Full AR PDF  : {AR_PDF.stat().st_size  // 1024 // 1024} MB  →  Subset: {AR_SUBSET.stat().st_size // 1024} KB")

⏭  Subset PDFs already exist — skipping extraction
   EN: 9419 KB
   AR: 2964 KB


## Step 3 — Upload Subset PDFs to PageIndex

`submit_document(file_path)` uploads the small subset PDF to the `/doc/` endpoint.  
PageIndex indexes each page **visually** — no OCR output from us needed at all.  

The call returns immediately with a `doc_id`; indexing runs in the background.  
We cache the `doc_id` so we never re-upload if the kernel restarts.

In [3]:
if DOC_CACHE.exists():
    doc_ids = json.loads(DOC_CACHE.read_text())
    print(f"⏭  Loaded doc IDs from cache:")
    for k, v in doc_ids.items():
        print(f"   {k}: {v}")
else:
    doc_ids = {}

    for label, pdf_path in [("english", EN_SUBSET), ("arabic", AR_SUBSET)]:
        size_kb = pdf_path.stat().st_size // 1024
        print(f"Uploading {label} subset ({pdf_path.name}, {size_kb} KB, {len(SAMPLED_PAGES)} pages)...")
        result = pi_client.submit_document(file_path=str(pdf_path))
        print(f"  Raw response: {result}")

        doc_id = (
            result.get("doc_id")
            or result.get("id")
            or result.get("document_id")
        )

        if doc_id:
            doc_ids[label] = doc_id
            print(f"  ✅ {label} doc_id: {doc_id}")
        else:
            print(f"  ⚠️  No doc_id found in response keys: {list(result.keys())}")

    if doc_ids:
        DOC_CACHE.write_text(json.dumps(doc_ids, indent=2))
        print(f"\nSaved to {DOC_CACHE}")

print(f"\nEnglish doc_id : {doc_ids.get('english', 'MISSING')}")
print(f"Arabic doc_id  : {doc_ids.get('arabic',  'MISSING')}")

⏭  Loaded doc IDs from cache:
   english: pi-cmp57ce4000u001qw6ueq8vz3
   arabic: pi-cmp57cg9n00u201qw4bkt2o1i

English doc_id : pi-cmp57ce4000u001qw6ueq8vz3
Arabic doc_id  : pi-cmp57cg9n00u201qw4bkt2o1i


## Step 4 — Inspect the Visual Reasoning Tree

`get_tree(doc_id)` returns the reasoning graph PageIndex built from the PDF pages.  
Each node represents a section PageIndex identified visually — without any OCR input from us.

In [4]:
en_doc_id = doc_ids.get("english")
ar_doc_id = doc_ids.get("arabic")

for label, doc_id in doc_ids.items():
    if not pi_client.is_retrieval_ready(doc_id):
        print(f"⏳ {label} ({doc_id}) not ready yet — re-run the polling cell above")
        continue

    print(f"\n{'='*60}")
    print(f"REASONING TREE — {label.upper()} (doc_id: {doc_id})")
    print(f"{'='*60}")

    tree       = pi_client.get_tree(doc_id, node_summary=True)
    raw_result = tree.get("result")

    if isinstance(raw_result, list):
        nodes = raw_result
    elif isinstance(raw_result, dict):
        nodes = raw_result.get("nodes") or raw_result.get("structure") or []
    else:
        nodes = []

    print(f"Total nodes: {len(nodes)}")
    for node in nodes[:8]:
        node_id    = node.get("node_id") or node.get("id", "?")
        title      = node.get("title")   or node.get("name", "(no title)")
        page_index = node.get("page_index", "?")
        summary    = (node.get("prefix_summary") or node.get("summary") or node.get("text", ""))[:100]
        print(f"  [{node_id}] page_index={page_index} | {title}")
        if summary:
            print(f"         {summary.replace(chr(10), ' ')}...")
    if len(nodes) > 8:
        print(f"  ... and {len(nodes)-8} more nodes")


REASONING TREE — ENGLISH (doc_id: pi-cmp57ce4000u001qw6ueq8vz3)
Total nodes: 9
  [0000] page_index=1 | Opening Statement
         This text is an opening statement from the Madinah Development Authority, outlining its commitment t...
  [0002] page_index=3 | Executive Summary
         This report provides a comprehensive analysis of Al Madinah City's livability, aiming to enhance its...
  [0007] page_index=5 | Population
         The text provides information on the population of Al Madinah Region and Governorate, including spec...
  [0008] page_index=6 | Easy and Efficient Accessibility to the Job
         The text highlights Al Madinah's accessibility to jobs, emphasizing its compact, mixed-use nature. I...
  [0009] page_index=7 | Al Madinah Bus Project
         # Al Madinah Bus Project  Al Madinah Bus is an important project that aims to serve the transportati...
  [0010] page_index=8 | AL Madinah Tomorrow
         # AL Madinah Tomorrow  Considering the high cost and the high energy

## Step 5 — Submit Query → Get Page References

`submit_query(doc_id, query)` returns a `retrieval_id`.  
`get_retrieval(retrieval_id)` gives the result — including which **pages** are most relevant.  

This is the core of the vision RAG idea: PageIndex acts as the **page selector**, not the answer generator.

In [5]:
import re
import json
import base64

# ── Image compression ─────────────────────────────────────────────────────────

def compress_image(img_bytes: bytes, max_width: int = 1024, quality: int = 82) -> bytes:
    """
    Resize (if wider than max_width) and re-encode as JPEG.
    Reduces a typical 150-DPI page PNG (~500 KB) to ~80-120 KB.
    Base64 overhead is ~33%, so 120 KB JPEG → ~160 KB base64, well under the 4 MB limit.
    """
    img = Image.open(io.BytesIO(img_bytes))
    if img.mode in ("RGBA", "P"):
        img = img.convert("RGB")
    if img.width > max_width:
        ratio    = max_width / img.width
        new_size = (max_width, int(img.height * ratio))
        img      = img.resize(new_size, Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    return buf.getvalue()


# ── PageIndex retrieval ───────────────────────────────────────────────────────

def get_relevant_pages(doc_id: str, question: str, poll_timeout: int = 60) -> dict:
    submit_result = pi_client.submit_query(doc_id=doc_id, query=question)
    retrieval_id  = (
        submit_result.get("retrieval_id")
        or submit_result.get("id")
        or submit_result.get("query_id")
    )
    if not retrieval_id:
        return submit_result
    for i in range(poll_timeout):
        result = pi_client.get_retrieval(retrieval_id)
        if result.get("status") == "completed" or result.get("retrieved_nodes"):
            print(f"  Ready after {i+1}s")
            return result
        if result.get("status") == "failed":
            return result
        time.sleep(1)
    return result


def extract_page_numbers(retrieval_result: dict) -> list:
    """Parse physical_index fields → subset page numbers."""
    nodes = retrieval_result.get("retrieved_nodes", [])
    pages = set()
    for node in nodes:
        for content_group in node.get("relevant_contents", []):
            items = content_group if isinstance(content_group, list) else [content_group]
            for item in items:
                if not isinstance(item, dict):
                    continue
                raw   = item.get("physical_index", "")
                match = re.search(r"physical_index_(\d+)", raw)
                if match:
                    pages.add(int(match.group(1)))
    result = sorted(pages)
    orig   = [SUBSET_PAGE_MAP.get(p, "?") for p in result]
    print(f"  Subset pages   : {result}")
    print(f"  Original pages : {orig}")
    return result


# ── Image loading ─────────────────────────────────────────────────────────────

def load_page_images(subset_page_numbers: list, img_dir: Path) -> list:
    """
    Remap subset page numbers → original page numbers → load PNGs → compress to JPEG.
    Returns list of (page_label, jpeg_bytes) tuples.
    """
    images = []
    for p in subset_page_numbers:
        orig = SUBSET_PAGE_MAP.get(p)
        if orig is None:
            print(f"  ⚠️  Subset page {p} not in SUBSET_PAGE_MAP — skipping")
            continue
        img_path = img_dir / f"page_{orig:03d}.png"
        if img_path.exists():
            raw_bytes  = img_path.read_bytes()
            jpeg_bytes = compress_image(raw_bytes)
            raw_kb  = len(raw_bytes)  // 1024
            jpeg_kb = len(jpeg_bytes) // 1024
            print(f"  ✅ subset p{p} → original p{orig} → {img_path.name}  ({raw_kb} KB → {jpeg_kb} KB JPEG)")
            images.append((f"p{orig}", jpeg_bytes))
        else:
            print(f"  ❌ Not found: {img_path}")
    return images


# ── Groq Vision answer generation ────────────────────────────────────────────

MAX_IMAGES_PER_REQUEST = 5   # Groq hard limit for vision requests

def answer_from_pages(question: str, page_images: list, language: str = "en") -> str:
    """
    Send question + up to 5 page images to Llama 4 Scout via Groq.

    Multiple images are passed as separate image_url entries in the same
    content array — same pattern as the official PageIndex vision RAG cookbook.
    Groq allows a maximum of 5 images per request (4 MB each after base64).
    """
    if not page_images:
        return "No page images to analyse."

    # Cap at Groq's hard limit
    batch = page_images[:MAX_IMAGES_PER_REQUEST]
    if len(page_images) > MAX_IMAGES_PER_REQUEST:
        skipped = [lbl for lbl, _ in page_images[MAX_IMAGES_PER_REQUEST:]]
        print(f"  ℹ️  Capped at {MAX_IMAGES_PER_REQUEST} images — skipping pages {skipped}")

    page_labels = [label for label, _ in batch]

    if language == "ar":
        prompt = (
            "أنت محلل خبير في تقارير التنمية الحضرية لمدينة المدينة المنورة لعام 2024. "
            f"الصفحات المرفقة {page_labels} اختارها PageIndex كأكثر الصفحات صلة. "
            "أجب بالعربية فقط. "
            "اذكر رقم الصفحة الأصلي لكل معلومة. إذا لم تجد الإجابة، قل ذلك صراحة.\n\n"
            f"السؤال: {question}"
        )
    else:
        prompt = (
            "You are an expert analyst of the 2024 Madinah Tranquil Livable City report. "
            f"Pages {page_labels} were selected by PageIndex as most relevant. "
            "Answer using ONLY what is visible in these page images. "
            "Cite the page label for every fact (e.g. [p.10]). "
            "If the answer is not visible, say so explicitly.\n\n"
            f"Question: {question}"
        )

    # Build content: text prompt first, then one image_url block per page
    content = [{"type": "text", "text": prompt}]
    for _, img_bytes in batch:
        b64      = base64.b64encode(img_bytes).decode("utf-8")
        data_uri = f"data:image/jpeg;base64,{b64}"
        content.append({
            "type"      : "image_url",
            "image_url" : {"url": data_uri},
        })

    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": content}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print("✅ All pipeline functions defined:")
print("   compress_image / get_relevant_pages / extract_page_numbers / load_page_images / answer_from_pages")
print(f"   Vision backend : Groq — {VISION_MODEL}")
print(f"   Max images/req : {MAX_IMAGES_PER_REQUEST}")

✅ All pipeline functions defined:
   compress_image / get_relevant_pages / extract_page_numbers / load_page_images / answer_from_pages
   Vision backend : Groq — meta-llama/llama-4-scout-17b-16e-instruct
   Max images/req : 5


## Step 6 — Page Mapping Reference

Reminder of how subset PDF page numbers (1-14) map back to original full-PDF page numbers and PNG filenames.

In [6]:
print("Subset page → original page mapping:")
for subset_p, orig_p in SUBSET_PAGE_MAP.items():
    print(f"  subset page {subset_p:>2}  →  original page {orig_p:>3}  →  page_{orig_p:03d}.png")

Subset page → original page mapping:
  subset page  1  →  original page   7  →  page_007.png
  subset page  2  →  original page   8  →  page_008.png
  subset page  3  →  original page  10  →  page_010.png
  subset page  4  →  original page  11  →  page_011.png
  subset page  5  →  original page  20  →  page_020.png
  subset page  6  →  original page  37  →  page_037.png
  subset page  7  →  original page  38  →  page_038.png
  subset page  8  →  original page  53  →  page_053.png
  subset page  9  →  original page  60  →  page_060.png
  subset page 10  →  original page  69  →  page_069.png
  subset page 11  →  original page  70  →  page_070.png
  subset page 12  →  original page  83  →  page_083.png
  subset page 13  →  original page 107  →  page_107.png
  subset page 14  →  original page 109  →  page_109.png


## Step 7 — Quick Pipeline Test (English + Arabic)

End-to-end test: PageIndex retrieval → page image loading → Gemini Vision answer.

In [7]:
Q_EN = "What are the livability indicators for Madinah in 2024?"
Q_AR = "ما هي مؤشرات قابلية العيش في المدينة المنورة؟"

print("=" * 65)
print(f"ENGLISH  Q: {Q_EN}")
print("=" * 65)
en_retrieval = get_relevant_pages(en_doc_id, Q_EN)
en_pages     = extract_page_numbers(en_retrieval)
en_images    = load_page_images(en_pages, IMG_EN_DIR)
en_answer    = answer_from_pages(Q_EN, en_images, language="en")
print(f"\n{en_answer}")

print()
print("=" * 65)
print(f"ARABIC   Q: {Q_AR}")
print("=" * 65)
ar_retrieval = get_relevant_pages(ar_doc_id, Q_AR)
ar_pages     = extract_page_numbers(ar_retrieval)
ar_images    = load_page_images(ar_pages, IMG_AR_DIR)
ar_answer    = answer_from_pages(Q_AR, ar_images, language="ar")
print(f"\n{ar_answer}")

ENGLISH  Q: What are the livability indicators for Madinah in 2024?
  Ready after 11s
  Subset pages   : [3, 13]
  Original pages : [10, 107]
  ✅ subset p3 → original p10 → page_010.png  (347 KB → 284 KB JPEG)
  ✅ subset p13 → original p107 → page_107.png  (271 KB → 173 KB JPEG)

The livability indicators for Madinah in 2024 are not explicitly listed on the provided pages. However, the sectors that contribute to the livability of Madinah City are [p10]:
1. **Spatial and Historical Perspective Sector**: This sector focuses on the city's culture, heritage, and history.
2. **A Humanized Livable City Sector of the City**: This sector prioritizes humanization by redesigning streets and public places.
3. **Urban Mobility Sector**: This sector includes the city's road network, public transportation, and traffic congestion.
4. **Affordable Decent Housing sector**: This sector focuses on housing price-to-income ratio, housing adequacy, and access to basic facilities.
5. **Sustainable Environmen

## Step 8 — Compare Vision vs Text Pipeline

Run the same questions that scored **partial** in Spike 05 and see if vision mode improves them.

In [8]:
TEST_QUESTIONS = [
    ("What are the livability indicators used in this report?",            "en", en_doc_id, IMG_EN_DIR),
    ("What is the population of Madinah according to the report?",         "en", en_doc_id, IMG_EN_DIR),
    ("What sustainability goals were achieved in 2024?",                   "en", en_doc_id, IMG_EN_DIR),
    ("Which neighborhoods scored highest on livability?",                  "en", en_doc_id, IMG_EN_DIR),
    ("What are the key challenges facing Madinah's urban development?",    "en", en_doc_id, IMG_EN_DIR),
    ("ما هي مؤشرات قابلية العيش المستخدمة في هذا التقرير؟",              "ar", ar_doc_id, IMG_AR_DIR),
    ("ما هو عدد سكان المدينة المنورة وفقاً للتقرير؟",                     "ar", ar_doc_id, IMG_AR_DIR),
    ("ما هي أبرز إنجازات التنمية المستدامة في عام 2024؟",                 "ar", ar_doc_id, IMG_AR_DIR),
]

vision_results = []

for i, (question, lang, doc_id, img_dir) in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*65}")
    print(f"Test {i+1}/{len(TEST_QUESTIONS)} [{lang.upper()}]")
    print(f"Q: {question}")
    print("-" * 65)

    # 1. Ask PageIndex which pages to look at
    retrieval  = get_relevant_pages(doc_id, question)
    pages      = extract_page_numbers(retrieval)
    print(f"PageIndex selected subset pages: {pages}")

    # 2. Load those page images (remapped to original page numbers)
    images = load_page_images(pages, img_dir)

    # 3. Gemini reads the images
    answer = answer_from_pages(question, images, language=lang)
    print(f"\nAnswer:\n{answer}")

    vision_results.append({
        "question"     : question,
        "lang"         : lang,
        "subset_pages" : pages,
        "orig_pages"   : [SUBSET_PAGE_MAP.get(p) for p in pages],
        "answer"       : answer,
    })


Test 1/8 [EN]
Q: What are the livability indicators used in this report?
-----------------------------------------------------------------
  Ready after 10s
  Subset pages   : [2, 3, 4, 13, 14]
  Original pages : [8, 10, 11, 107, 109]
PageIndex selected subset pages: [2, 3, 4, 13, 14]
  ✅ subset p2 → original p8 → page_008.png  (432 KB → 129 KB JPEG)
  ✅ subset p3 → original p10 → page_010.png  (347 KB → 284 KB JPEG)
  ✅ subset p4 → original p11 → page_011.png  (352 KB → 289 KB JPEG)
  ✅ subset p13 → original p107 → page_107.png  (271 KB → 173 KB JPEG)
  ✅ subset p14 → original p109 → page_109.png  (319 KB → 198 KB JPEG)

Answer:
The livability indicators used in this report are not explicitly listed on the provided pages. However, based on the chapter titles and content, the following indicators can be inferred [p8]:
* Spatial and Historical Perspective Sector
* A Humanized Livable City Sector
* Urban Mobility Sector
* Affordable Decent Housing sector
* Sustainable Environment Sector

## Step 9 — Score and Summarise

In [9]:
# Fill in after reading answers above
# Options: 'correct', 'partial', 'wrong', 'unsure'
SCORES = [
    'correct',  # Q1 EN — livability indicators
    'unsure',  # Q2 EN — population
    'correct',  # Q3 EN — sustainability goals 2024
    'correct',  # Q4 EN — neighborhoods
    'correct',  # Q5 EN — key challenges
    'partial',  # Q1 AR — livability indicators
    'correct',  # Q2 AR — population
    'correct',  # Q3 AR — sustainability goals
]

from collections import Counter
score_counts = Counter(SCORES)
total   = len(SCORES)
correct = score_counts['correct']
partial = score_counts['partial']
wrong   = score_counts['wrong']
unsure  = score_counts['unsure']
scored  = total - unsure

print("=" * 55)
print("SPIKE 06 — VISION RAG RESULTS")
print("=" * 55)
print(f"Total questions : {total}")
print(f"Correct         : {correct}  ({correct/total*100:.0f}%)")
print(f"Partial         : {partial}  ({partial/total*100:.0f}%)")
print(f"Wrong           : {wrong}   ({wrong/total*100:.0f}%)")
print(f"Unsure          : {unsure}")
print()

eff = (correct + 0.5 * partial) / scored * 100 if scored > 0 else 0
print(f"Effective score  : {eff:.0f}%")
print(f"Spike 05 baseline: 81%")
print()

if eff > 81:
    delta = eff - 81
    print(f"VISION RAG IS BETTER (+{delta:.0f}%)")
    print("  Adopt vision-based pipeline as the primary approach.")
    print("  The page images carry richer information than OCR text.")
elif eff >= 70:
    print(f"VISION RAG IS COMPARABLE ({eff:.0f}% vs 81%)")
    print("  Consider hybrid: use vision for tables/charts, text for dense paragraphs.")
else:
    print(f"VISION RAG IS WEAKER ({eff:.0f}% vs 81%)")
    print("  Possible causes: PageIndex vision indexing needs time, or page numbers off by 1.")
    print("  Check: Are the page images sharp enough? Did indexing fully complete?")

SPIKE 06 — VISION RAG RESULTS
Total questions : 8
Correct         : 6  (75%)
Partial         : 1  (12%)
Wrong           : 0   (0%)
Unsure          : 1

Effective score  : 93%
Spike 05 baseline: 81%

VISION RAG IS BETTER (+12%)
  Adopt vision-based pipeline as the primary approach.
  The page images carry richer information than OCR text.
